# 01 Trimodal VAMPnet

How the three input blocks are built and fed to the network, the training call, and inference with the
saved models. The feature files are not part of this folder (about 10 GB); set the three paths below to
your copies.

| block | content | branch |
|---|---|---|
| distance | 31 x 31 C$\alpha$ distances, incoming peptide vs growth end | CNN |
| dihedral | sin/cos of backbone $\phi,\psi$ of both chains (244) | MLP |
| H-bond | bitmask of the longest-lived interfacial H-bonds (86 $\nu$, 75 S) | MLP |

The saved weights fix the state indices used in every later notebook. Retraining gives a different
numbering.

In [ ]:
import numpy as np
import torch
import MDAnalysis as mda

import abeta_tips as T
import trimodal_vampnet as V

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
FOLD = "nu_8EZD"          # or "S_7Q4M"
F = T.FOLDS[FOLD]
MODEL_DIR = f"../vampnet_models/{FOLD}"

# inputs not shipped here
DIST_DIH_PKL = "/path/to/Features/deeptime_dist_dih.pkl"     # list of (T, 1205) arrays, one per trajectory
HBOND_NPZ = "/path/to/filtered_hbond_dtraj.npz"               # traj_0, traj_1, ... (T, n_hbond) bitmasks

## Distance and dihedral features

`featurize` gives the 1205 columns of one trajectory (961 distances + 244 dihedral terms). It was run on
every trajectory in the order of `T.FOLDS[FOLD]["batches"]`; the lists were pickled as `DIST_DIH_PKL`.

In [ ]:
# example on one trajectory
# u = mda.Universe("em.gro", "traj_0.xtc")
# x = V.featurize(u, F["peptide_ca"], F["growth_ca"], F["dihedral_sel"])     # (T, 1205)

## Assemble the trimodal input

The saved scaler (min-max to [-1, 1], fitted once over all frames) is applied to the distance and dihedral
columns together, then the block is split: distances reshaped to 31 x 31 for the CNN, dihedrals kept as a
vector. The H-bond bitmask is used as stored.

In [ ]:
dist, dih, hbond = V.load_inputs(DIST_DIH_PKL, f"{MODEL_DIR}/dist_dih_scaler.pkl", HBOND_NPZ)
print(len(dist), dist[0].shape, dih[0].shape, hbond[0].shape)
assert hbond[0].shape[1] == F["n_hbond"]

## Training (as used for the manuscript models)

Lag 5 frames (0.5 ns), K = 6, latent dimension 8 per branch, 30 epochs. Do not run this if you want the
published state numbering; load the saved weights below instead.

In [ ]:
RETRAIN = False
if RETRAIN:
    lobe, train_scores, val_scores = V.train(dist, dih, hbond, device)
    torch.save(lobe.state_dict(), "trimodal_vampnet_lobe_state_dict.pt")

In [ ]:
import matplotlib.pyplot as plt

tr = np.load(f"{MODEL_DIR}/vampnet_train_scores.npy")
va = np.load(f"{MODEL_DIR}/vampnet_validation_scores.npy")
plt.loglog(*tr.T, c="k", label="training")
plt.loglog(*va.T, c="r", label="validation")
plt.xlabel("step"); plt.ylabel("VAMP-2"); plt.legend()
print(f"final VAMP-2: train {tr[-1, 1]:.4f}, validation {va[-1, 1]:.4f}")

## Inference with the saved model

In [ ]:
lobe = V.load_lobe(f"{MODEL_DIR}/trimodal_vampnet_lobe_state_dict.pt", F["n_hbond"], device)
probs = V.predict(lobe, dist, dih, hbond, device)
states = probs.argmax(1)

saved = np.load(f"../saved_data/{FOLD}/trimodal_state_assignments.npy")
print("agreement with saved assignments:", (states == saved).mean())
print("frames with max probability > 0.8:", round((probs.max(1) > 0.8).mean(), 3))
H = -(probs * np.log(probs + 1e-12)).sum(1)
print("mean assignment entropy:", round(H.mean(), 3), " (uniform =", round(np.log(6), 3), ")")